# Agent的初步构建

主要使用工具 

    LangGraph, LangChain, Travily, 高德


In [1]:
import os 
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv("apikey.env")
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_PROJECT'] = "My Agent"
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY-API-KEY")
gaode_key = os.getenv("GAODEWEATHER_API_KEY")


创建工具并且进行包装

In [2]:
from langchain_tavily import TavilySearch
from langchain_core.tools import StructuredTool
from datetime import datetime
import requests

search_tool = TavilySearch(max_results=5)

def get_current_time(dummy: str = None) -> dict:
    "获取当前日期和时间"
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return {"current_time": now}

time_tool = StructuredTool.from_function(
    func=get_current_time,
    name="get_current_time",
    description="如果你想知道对话发生的当前的日期和时间，请使用这个工具",
)


def get_adcode(keyword: str, api_key: str = gaode_key) -> dict:
    """
    根据地点关键词获取高德的 adcode
    参数:
        keyword: 地点名称，例如 "成都"
        api_key: 高德开放平台 API key
    返回:
        包含 adcode 的字典，例如 {"adcode": "510100"}
    """
    url = "https://restapi.amap.com/v3/config/district"
    params = {
        "key": api_key,
        "keywords": keyword,
        "subdistrict": 0  # 先不获取下级行政区
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        # 检查请求状态
        if data.get("status") == "1" and data.get("districts"):
            # 返回第一个匹配地区的adcode
            adcode = data["districts"][0]["adcode"]
            return {"adcode": adcode, "keyword": keyword}
        else:
            return {
                "error": data.get("info", "查询失败或未找到地区"),
                "keyword": keyword
            }
    except Exception as e:
        return {"error": str(e), "keyword": keyword}
    
adcode_tool = StructuredTool.from_function(
    func=get_adcode,
    name="get_adcode",
    description="用于查询天气的前置：如果用户需要知道adcode必须首先使用这个工具进行查询"
)


def get_weather_by_adcode(adcode: str, 
                          api_key: str = gaode_key, 
                          extensions: str ="all") -> dict:
    """
    根据高德行政区 adcode 查询天气信息。
    参数:
        adcode: 行政区划代码
        api_key: 高德开放平台的 API Key
        extensions: 
            - "base" 表示实况天气 
            - "all" 表示预报天气
    返回:
        包含天气信息的字典，例如:
        {"city": "成都", "weather": "多云", "temperature": "22", ...}
    """
    url = "https://restapi.amap.com/v3/weather/weatherInfo"
    params = {
        "key": api_key,
        "city": adcode,
        "extensions": extensions
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        if data.get("status") == "1":
            if extensions == "base":
                # 解析实况天气
                lives = data.get("lives", [])
                if lives:
                    live = lives[0]
                    return {
                        "province": live.get("province"),
                        "city": live.get("city"),
                        "weather": live.get("weather"),
                        "temperature": live.get("temperature"),
                        "winddirection": live.get("winddirection"),
                        "windpower": live.get("windpower"),
                        "humidity": live.get("humidity"),
                        "reporttime": live.get("reporttime")
                    }
            elif extensions == "all":
                # 解析预报天气
                forecasts = data.get("forecasts", [])
                if forecasts:
                    # 这里可以处理预报数据，通常会包含未来几天的天气
                    forecast = forecasts[0]
                    return forecast
            return {"error": "未找到天气数据", "adcode": adcode, "extensions": extensions}
        else:
            return {"error": data.get("info", "查询失败"), "adcode": adcode}
    except Exception as e:
        return {"error": str(e), "adcode": adcode}

get_weather_tool = StructuredTool.from_function(
    func=get_weather_by_adcode,
    name="get_weather_by_adcode",
    description="如果用户需要查询实时的天气，必须首先需要调用get_adcode工具获得adcode在调用"
)

In [3]:
tools = [time_tool, adcode_tool, get_weather_tool, search_tool]
tools

[StructuredTool(name='get_current_time', description='如果你想知道对话发生的当前的日期和时间，请使用这个工具', args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x00000275E51D8AF0>),
 StructuredTool(name='get_adcode', description='用于查询天气的前置：如果用户需要知道adcode必须首先使用这个工具进行查询', args_schema=<class 'langchain_core.utils.pydantic.get_adcode'>, func=<function get_adcode at 0x0000027598EDD090>),
 StructuredTool(name='get_weather_by_adcode', description='如果用户需要查询实时的天气，必须首先需要调用get_adcode工具获得adcode在调用', args_schema=<class 'langchain_core.utils.pydantic.get_weather_by_adcode'>, func=<function get_weather_by_adcode at 0x0000027598EDD240>),
 TavilySearch(max_results=5, api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]

In [4]:
model = ChatDeepSeek(base_url=BASE_URL, api_key=API_KEY,
                     temperature=0.00,
                     model="deepseek-chat")

In [32]:
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage

tool_usage_prompt = """
    你是一个擅长运用各种工具进行信息检索、数据分析和逻辑推理的专家。
    你的目标是提供准确、完整、有深度的答案。
    在开始之前，请先先确认这个问题是否需要使用工具。
    如果需要工具：规划为解决这个问题可能需要使用的工具，并简要说明选择每个工具的理由。
    使用对应的工具进，最后把收集到的内容保持原样输出，并最后附上一句精简的总结。
    """
tool_agent = create_react_agent(
    model=model,
    tools=tools,
    prompt=tool_usage_prompt
)

def decide_nedd_tool(state):
    
    question = state["question"]
    decide_prompt = f"""
    你是一个智能工具决策助手。你有以下的工具：
    
    StructuredTool(name='get_current_time', description='如果你想知道对话发生的当前的日期和时间，请使用这个工具')
    StructuredTool(name='get_adcode', description='用于查询天气的前置：如果用户需要知道adcode必须首先使用这个工具进行查询')
    StructuredTool(name='get_weather_by_adcode', description='如果用户需要查询实时的天气，必须首先需要调用get_adcode工具获得adcode在调用')
    TavilySearch(max_results=5)

    你的任务是根据用户请求，判断用户的请求{question}是否需要使用工具。
    需要工具就输出为tool
    """
    decision = model.invoke(decide_prompt)
    state["need_tools"] = "tool" in decision.content.lower()
    return state

def use_tool(state):
    # 初始化循环计数器
    if "retry_count" not in state:
        state["retry_count"] = 0
    
    # 增加重试次数
    state["retry_count"] += 1

    question = state["question"]
    suggestion = state.get("reflection_suggestion", "")

    if suggestion:
        question = f"{question}\n请根据以下建议调整工具调用策略：{suggestion}"

    mg = {"messages": [HumanMessage(content=state["question"])]}
    result = tool_agent.invoke(mg)

    # 保留历史记录
    current_result = result["messages"][-1].content
    
    # 保留历史记录到新字段
    if "tool_result_history" not in state:
        state["tool_result_history"] = []  # 初始化历史记录列表
    
    # 添加当前结果到历史记录
    state["tool_result_history"].append(current_result)
    
    # 保持 tool_result 作为最新结果的字符串
    state["tool_result"] = current_result

    return state

def reflection(state):
    reflection_prompt = f"""
    根据用户的提问{state["question"]}回答和
    收集到的信息{state.get("tool_result", "")},{state.get("tool_result_history", "")}
    现在，请对你的初步答案进行审查，包括但不限于：

    1. 工具选择是否正确？是否反复调用？
    2. 是否需要重新调用工具？
    3. 如果需要，请给出改进建议。

    请用保证 JSON 格式输出，字段如下：
    {{
    "reflection_decision": "retry"/"next",
    "reflection_suggestion": "string"
    }}
    """
    # 检查重试次数，避免无限循环
    max_retries = 2  # 设置最大重试次数
    retry_count = state.get("retry_count", 0)
    if retry_count >= max_retries:
        state["reflection_decision"] = "next"  
        return state
    
    decision = model.invoke(reflection_prompt)
    state["reflection_decision"] = decision.content
    return state


def summarize(state):
    summarize_prompt = f"""
    你是一名专业的资深编辑员，请你根据用户的输入和搜索到的内容(如果存在)解析并进行推理总结。
    注意，可以结合你的知识库进行回答，
    但请不要修改客观事实，如若需要可以在不修改参考资料/原意下适当添加主观表达。
    如果用户有需要结构化输出的格式，请按照用户的意图进行格式化输出。
    只输出你的答案。

    这是用户的问题{state["question"]},
    这里是参考资料{state.get("tool_result", "")}以及{state.get("tool_result_history","")}
    """
    ans = model.invoke(summarize_prompt)
    state["final_answer"] = ans.content
    return state

In [33]:
workflow = StateGraph(dict)

workflow.add_node("decide", decide_nedd_tool)
workflow.add_node("tool", use_tool)
workflow.add_node("reflection", reflection)
workflow.add_node("summarize", summarize)

workflow.add_conditional_edges(
    "decide",
    lambda s: "tool" if s["need_tools"] else "summarize",
    {"tool": "tool", "summarize": "summarize"}
)

# reflection 分支
workflow.add_edge("tool", "reflection")
workflow.add_conditional_edges(
    "reflection",
    lambda s: "tool" if "retry" in str(s.get("reflection_decision", "")).lower() else "summarize",
    {"tool": "tool", "summarize": "summarize"}
)

workflow.add_edge("summarize", END)
workflow.set_entry_point("decide")

my_agent = workflow.compile()

In [34]:
result = my_agent.invoke({"question": "今天是星期几？"})
print(result["final_answer"])

今天是星期六。


In [8]:
result

{'question': '今天是星期几？',
 'need_tools': True,
 'retry_count': 1,
 'tool_result_history': ['今天是**星期五**。\n\n根据获取的时间信息，当前是2025年10月10日，星期五。'],
 'tool_result': '今天是**星期五**。\n\n根据获取的时间信息，当前是2025年10月10日，星期五。',
 'reflection_decision': '{\n    "reflection_decision": "next",\n    "reflection_suggestion": "工具选择正确且仅调用一次，已准确获取当前日期（2025年10月10日）和星期信息（星期五），答案完整清晰，无需重新调用工具。"\n}',
 'final_answer': '今天是**星期五**。'}

In [9]:
for event in my_agent.stream({"question": "我明天想去广东的汕头玩，请你看一下天气情况和准备一下出行计划。"}):
    print(event)
    if "summarize" in event and "final_answer" in event["summarize"]:
        print("\n🧭 最终总结：", event["summarize"]["final_answer"])

{'decide': {'question': '我明天想去广东的汕头玩，请你看一下天气情况和准备一下出行计划。', 'need_tools': True}}
{'tool': {'question': '我明天想去广东的汕头玩，请你看一下天气情况和准备一下出行计划。', 'need_tools': True, 'retry_count': 1, 'tool_result_history': ['基于查询到的信息，我为您准备了详细的汕头出行计划：\n\n## 汕头出行计划（明天）\n\n### 🌤️ 天气情况\n**明天（10月11日，周六）天气：**\n- **白天：** 晴，最高温度33°C\n- **夜间：** 晴，最低温度27°C  \n- **风向：** 东南风1-3级\n- **天气状况：** 晴朗舒适，适合户外活动\n\n**未来几天天气趋势：**\n- 周日：晴转阵雨，温度27-33°C\n- 周一：晴转多云，温度27-32°C\n\n### 🎯 出行建议\n\n**衣物准备：**\n- 轻薄夏装（短袖、短裤、裙子）\n- 防晒用品（帽子、太阳镜、防晒霜）\n- 舒适的步行鞋\n- 雨伞（周日可能有阵雨）\n\n### 🏞️ 推荐景点\n\n**1. 汕头小公园**\n- 汕头老城核心地标，民国建筑风格\n- 必游：中山纪念亭、南生百货大楼、汕头开埠文化陈列馆\n- 附近美食：小公园蛋挞、爱西干面、侗平肠粉\n\n**2. 东海岸公园**\n- 13公里无敌海岸线，被称为"汕头迈亚密"\n- 四个园区：时间之环、时间溪谷、梦想腾飞、拥抱未来\n- 适合：运动、野餐、放风筝、欣赏日落\n\n**3. 礐石风景区**\n- 汕头最美后花园，自然风光优美\n\n**4. 南澳岛**\n- 海岛风光，青澳湾沙滩\n\n### 🍽️ 美食推荐\n\n**必吃美食：**\n1. **牛肉火锅** - 杏花吴记、八合里、福合埕\n2. **潮汕生腌** - 瑞娇嫲嫲、金二顺\n3. **卤水鹅** - 日日香卤鹅饭店\n4. **粽球** - 小公园老牌粽球\n5. **粿条汤** - 老胡牛肉粿\n\n**特色餐厅：**\n- 杏花吴记牛肉火锅（汕头牛肉火锅第一名）\n- 福合埕牛肉丸（60年老字号）\n- 吴记富苑（消夜打冷天花板）\n- 瑞娇嫲嫲潮汕生腌（明

In [11]:
for event in my_agent.stream({"question": "我明天想去广东的广州玩两三天，请你看一下天气情况和准备一下出行计划。"}):
    print(event)
    if "summarize" in event and "final_answer" in event["summarize"]:
        print("\n🧭 最终总结：", event["summarize"]["final_answer"])

{'decide': {'question': '我明天想去广东的广州玩两三天，请你看一下天气情况和准备一下出行计划。', 'need_tools': True}}
{'tool': {'question': '我明天想去广东的广州玩两三天，请你看一下天气情况和准备一下出行计划。', 'need_tools': True, 'retry_count': 1, 'tool_result_history': ['基于查询到的天气信息和旅游攻略，我为您制定了一份详细的广州三日游出行计划：\n\n## 广州三日游出行计划\n\n### 📅 天气情况（2025年10月11日-13日）\n\n**明天（10月11日，周六）**：\n- 白天：晴，35°C，东风1-3级\n- 夜间：雷阵雨，25°C\n\n**后天（10月12日，周日）**：\n- 全天：雷阵雨，34°C/24°C，东南风1-3级\n\n**第三天（10月13日，周一）**：\n- 全天：雷阵雨，33°C/24°C，东南风1-3级\n\n### 🎯 出行建议\n- **衣物准备**：轻便夏装为主，带雨伞和薄外套\n- **防晒防雨**：白天炎热需防晒，下午到晚上可能有雷阵雨\n- **舒适度**：温度较高，注意补水防暑\n\n### 🗺️ 三日游行程安排\n\n#### **第一天：历史文化探索**\n**上午**：\n- **越秀公园**（免费）：广州最大公园，看五羊石像、镇海楼\n- **广州博物馆**：了解广州历史\n\n**下午**：\n- **中山纪念堂**（10元）：宏伟八角形建筑，了解孙中山历史\n- **南越王博物院**（10元）：西汉南越王墓，看丝缕玉衣\n\n**晚上**：\n- **花城广场**：拍摄广州塔夜景，欣赏现代都市风光\n\n#### **第二天：欧陆风情与现代都市**\n**上午**：\n- **沙面岛**（免费）：欧式建筑群，摄影天堂\n- **圣心大教堂**（免费）：哥特式建筑，注意着装要求\n\n**下午**：\n- **陈家祠**（10元）：岭南建筑精华，精美雕刻艺术\n- **上下九步行街**：品尝地道粤式美食\n\n**晚上**：\n- **珠江夜游**（天字码头）：欣赏广州塔、猎德大桥灯光秀\n\n#### **第三天：自然风光与休闲购物**\n**上午**：\n-

In [16]:
for event in my_agent.stream({"question": "明天是星期几？"}):
    print(event)
    if "summarize" in event and "final_answer" in event["summarize"]:
        print("\n🧭 最终总结：", event["summarize"]["final_answer"])

{'decide': {'question': '明天是星期几？', 'need_tools': True}}
{'tool': {'question': '明天是星期几？', 'need_tools': True, 'retry_count': 1, 'tool_result_history': ['根据当前时间2025年10月10日（星期五），明天是2025年10月11日，也就是**星期六**。'], 'tool_result': '根据当前时间2025年10月10日（星期五），明天是2025年10月11日，也就是**星期六**。'}}
{'reflection': {'question': '明天是星期几？', 'need_tools': True, 'retry_count': 1, 'tool_result_history': ['根据当前时间2025年10月10日（星期五），明天是2025年10月11日，也就是**星期六**。'], 'tool_result': '根据当前时间2025年10月10日（星期五），明天是2025年10月11日，也就是**星期六**。', 'reflection_decision': '{\n    "reflection_decision": "next",\n    "reflection_suggestion": "工具选择正确，仅需一次调用即可根据当前日期（2025年10月10日星期五）准确推算出明天是星期六。无需重新调用工具。"\n}'}}
{'summarize': {'question': '明天是星期几？', 'need_tools': True, 'retry_count': 1, 'tool_result_history': ['根据当前时间2025年10月10日（星期五），明天是2025年10月11日，也就是**星期六**。'], 'tool_result': '根据当前时间2025年10月10日（星期五），明天是2025年10月11日，也就是**星期六**。', 'reflection_decision': '{\n    "reflection_decision": "next",\n    "reflection_suggestion": "工具选择正确，仅需一次调用即可根据当前日期（2025年10

In [43]:
for event in my_agent.stream({"question": "我后天需要到新疆出差，可以看一下当地的情况吗，以及有什么注意事项比如风俗习惯之类的？"}):
    print(event)
    if "decide" in event:
        print("="*80,"\n")
        print("\n模型正在判断是否进行工具调用")
        if "need_tools" in event["decide"]:
            print("\n用户：", event["decide"]["question"])
            print("\n工具调用决策：", event["decide"]["need_tools"])

    if "tool" in event:
        print("="*80,"\n")
        print("\n模型正在使用工具...")
        if "tool_result" in event["tool"]:
            print("\n模型检索结果/资料：", event["tool"]["tool_result"])
    if "reflection" in event:
        print("="*80,"\n")
        print("\n模型检查结果/资料中...")
        if "reflection_decision" in event["reflection"]:
            print("\n", event["reflection"]["reflection_decision"])
    if "summarize" in event:
        print("="*80,"\n")
        print(event["summarize"].keys())
        if "final_answer" in event["summarize"]:
            print("\n🧭 最终总结：", event["summarize"]["final_answer"])

{'decide': {'question': '我后天需要到新疆出差，可以看一下当地的情况吗，以及有什么注意事项比如风俗习惯之类的？', 'need_tools': True}}


模型正在判断是否进行工具调用

用户： 我后天需要到新疆出差，可以看一下当地的情况吗，以及有什么注意事项比如风俗习惯之类的？

工具调用决策： True
{'tool': {'question': '我后天需要到新疆出差，可以看一下当地的情况吗，以及有什么注意事项比如风俗习惯之类的？', 'need_tools': True, 'retry_count': 1, 'tool_result_history': ['基于收集到的信息，我来为您总结新疆出差的情况和注意事项：\n\n## 新疆天气情况（10月13日）\n- **天气**：多云，温度6-17℃\n- **风向**：北风1-3级\n- **特点**：早晚温差大，建议采用"洋葱式"穿衣法\n\n## 重要注意事项\n\n### 1. 证件准备\n- 身份证必须随身携带\n- 如前往白哈巴、塔什库尔干等边境地区需办理边防证\n\n### 2. 风俗习惯与商务礼仪\n**尊重当地文化：**\n- 参观清真寺等宗教场所时穿着得体，保持安静\n- 避免公开讨论宗教和政治话题\n- 尊重当地居民的私人空间，不要随意进入帐篷或房屋\n\n**饮食文化：**\n- 新疆以清真食品为主，避免在公共场所食用猪肉或饮酒\n- 尊重当地饮食习惯，不要浪费食物\n- 践踏粮食、食物和盐被认为会带来厄运\n\n**服饰礼仪：**\n- 与当地居民沟通时注意用语\n- 拍照或合影前先征得对方同意\n\n### 3. 实用建议\n- **时差**：新疆与北京时间有2小时时差\n- **防晒保湿**：气候干燥，紫外线强，注意防晒和保湿\n- **交通**：新疆地域广阔，景点间距离较远，合理安排行程\n- **安全**：新疆旅行很安全，但仍需保持警觉\n\n### 4. 商务交流\n- 初次见面时保持礼貌和尊重\n- 商务洽谈时避免过于直接的表达方式\n- 了解当地合作伙伴的民族背景和习俗\n\n**总结：新疆出差需特别注意尊重当地民族文化和宗教信仰，做好证件准备，合理安排行程，并注意气候适应。**'], 'tool_result': '基于收集到的信息，我来为您

In [30]:
s = {'summarize': {'question': '我明天想去广东的广州玩两三天，请你看一下天气情况和准备一下出行计划。', 'need_tools': True, 'retry_count': 2, 'tool_result_history': ['基于查询到的天气信息和旅游攻略，我为您制定了一份详细的广州三日游出行计划：\n\n## 广州三日游出行计划\n\n### 📅 天气情况（2025年10月11日-13日）\n\n**明天（10月11日，周六）**：\n- 白天：晴，35°C，东风1-3级\n- 夜间：雷阵雨，25°C\n\n**后天（10月12日，周日）**：\n- 全天：雷阵雨，34°C/24°C，东南风1-3级\n\n**第三天（10月13日，周一）**：\n- 全天：雷阵雨，33°C/24°C，东南风1-3级\n\n### 🎯 出行建议\n- **衣物准备**：轻便夏装为主，带雨伞和薄外套\n- **防晒防雨**：白天炎热需防晒，下午到晚上可能有雷阵雨\n- **舒适度**：温度较高，注意补水防暑\n\n### 🗺️ 三日游行程安排\n\n#### **第一天：历史文化探索**\n**上午**：\n- **越秀公园**（免费）：广州最大公园，看五羊石像、镇海楼\n- **广州博物馆**：了解广州历史\n\n**下午**：\n- **中山纪念堂**（10元）：宏伟八角形建筑，了解孙中山历史\n- **南越王博物院**（10元）：西汉南越王墓，看丝缕玉衣\n\n**晚上**：\n- **花城广场**：拍摄广州塔夜景，欣赏现代都市风光\n\n#### **第二天：欧陆风情与现代都市**\n**上午**：\n- **沙面岛**（免费）：欧式建筑群，摄影天堂\n- **圣心大教堂**（免费）：哥特式建筑，注意着装要求\n\n**下午**：\n- **陈家祠**（10元）：岭南建筑精华，精美雕刻艺术\n- **上下九步行街**：品尝地道粤式美食\n\n**晚上**：\n- **珠江夜游**（天字码头）：欣赏广州塔、猎德大桥灯光秀\n\n#### **第三天：自然风光与休闲购物**\n**上午**：\n- **白云山**：登山俯瞰广州全景，呼吸新鲜空气\n- **广东省博物馆**（免费，需预约）：了解岭南文化\n\n**下午**：\n- **北京路步行街**：逛千年古道遗址，看大佛寺夜景\n- **永庆坊**：西关老街区，体验传统岭南文化\n\n**晚上**：\n- **太古仓码头**：看日落，江边餐厅享受晚餐\n\n### 🍜 美食推荐\n- **银记肠粉**：地道肠粉\n- **南信牛奶甜品**：双皮奶等甜品\n- **广式烤鸭**：传统粤菜\n- **点心**：正宗广式点心\n\n### 💡 实用贴士\n1. **交通**：广州地铁发达，建议购买地铁日票\n2. **预约**：广东省博物馆需提前预约\n3. **着装**：参观教堂需避免短裤吊带\n4. **时间安排**：避开中午高温时段，合理安排室内外活动\n5. **预算**：景点门票约50-100元，餐饮人均50-100元/天\n\n这个行程结合了广州的历史文化、现代都市和自然风光，让您充分体验这座城市的多元魅力。祝您旅途愉快！', '基于查询到的天气信息和旅游攻略，我为您制定了一份详细的广州三日游出行计划：\n\n## 🌤️ 广州天气情况（10月11日-13日）\n\n**10月11日（周六）**：晴转雷阵雨，35°C/25°C，东风1-3级\n**10月12日（周日）**：雷阵雨，34°C/24°C，东南风1-3级\n**10月13日（周一）**：雷阵雨，33°C/24°C，东南风1-3级\n\n**天气提醒**：白天炎热，注意防晒；下午到晚上可能有雷阵雨，建议携带雨具。\n\n## 🗺️ 广州三日游行程规划\n\n### 第一天：历史文化探索日\n**上午**：\n- **越秀公园**（广州最大城市公园，有著名的五羊石像）\n- **广州博物馆**（了解广州历史）\n- **中山纪念堂**（纪念孙中山先生）\n\n**下午**：\n- **陈家祠**（岭南建筑艺术精品）\n- **永庆坊**（文艺街区，老字号美食云集）\n\n**晚上**：\n- **珠江夜游**（从天字码头出发，欣赏广州塔夜景）\n- **北京路步行街**（千年古道遗址，夜景如《千与千寻》）\n\n### 第二天：现代都市与自然风光\n**上午**：\n- **广州塔**（小蛮腰，登塔俯瞰全城）\n- **花城广场**（现代建筑群，拍照打卡）\n\n**下午**：\n- **沙面岛**（欧式建筑，宁静小岛）\n- **圣心大教堂**（石室圣心大教堂）\n\n**晚上**：\n- **大佛寺**（红墙黄瓦夜景，金碧辉煌）\n- **上下九步行街**（传统商业街，品尝地道粤菜）\n\n### 第三天：自然风光与美食体验\n**上午**：\n- **白云山**（广州名山，登山观景）\n- **云台花园**（春日花海）\n\n**下午**：\n- **长隆野生动物世界**（可选，如时间充裕）\n- **东山口**（文艺街区，特色小店）\n\n**晚上**：\n- **老字号饮茶**（北园酒家、泮溪酒家等）\n- **特色粤菜**（九大簋家宴等）\n\n## 🍽️ 美食推荐\n- **早茶**：北园酒家、泮溪酒家、南园酒家\n- **粤菜**：九大簋家宴、侨美食家\n- **小吃**：荔银肠粉、上下九步行街小吃\n- **甜品**：特色糖水、点心\n\n## 🚇 交通建议\n- 主要使用地铁出行（广州地铁网络发达）\n- 下载"广州地铁"APP方便查询路线\n- 景点间距离较近的可步行或共享单车\n\n## 🎒 出行准备\n- **衣物**：轻薄夏装，带件薄外套防空调\n- **雨具**：雨伞或雨衣（雷阵雨频繁）\n- **防晒**：防晒霜、帽子、太阳镜\n- **药品**：防暑药品、肠胃药\n- **其他**：充电宝、舒适鞋子\n\n## 💡 实用贴士\n1. 避开周末高峰期游览热门景点\n2. 提前预订热门餐厅和景点门票\n3. 注意防暑降温，多补充水分\n4. 雷阵雨多在下午，可安排室内活动\n5. 使用手机支付（微信/支付宝）更方便\n\n祝您在广州玩得愉快！'], 'tool_result': '基于查询到的天气信息和旅游攻略，我为您制定了一份详细的广州三日游出行计划：\n\n## 🌤️ 广州天气情况（10月11日-13日）\n\n**10月11日（周六）**：晴转雷阵雨，35°C/25°C，东风1-3级\n**10月12日（周日）**：雷阵雨，34°C/24°C，东南风1-3级\n**10月13日（周一）**：雷阵雨，33°C/24°C，东南风1-3级\n\n**天气提醒**：白天炎热，注意防晒；下午到晚上可能有雷阵雨，建议携带雨具。\n\n## 🗺️ 广州三日游行程规划\n\n### 第一天：历史文化探索日\n**上午**：\n- **越秀公园**（广州最大城市公园，有著名的五羊石像）\n- **广州博物馆**（了解广州历史）\n- **中山纪念堂**（纪念孙中山先生）\n\n**下午**：\n- **陈家祠**（岭南建筑艺术精品）\n- **永庆坊**（文艺街区，老字号美食云集）\n\n**晚上**：\n- **珠江夜游**（从天字码头出发，欣赏广州塔夜景）\n- **北京路步行街**（千年古道遗址，夜景如《千与千寻》）\n\n### 第二天：现代都市与自然风光\n**上午**：\n- **广州塔**（小蛮腰，登塔俯瞰全城）\n- **花城广场**（现代建筑群，拍照打卡）\n\n**下午**：\n- **沙面岛**（欧式建筑，宁静小岛）\n- **圣心大教堂**（石室圣心大教堂）\n\n**晚上**：\n- **大佛寺**（红墙黄瓦夜景，金碧辉煌）\n- **上下九步行街**（传统商业街，品尝地道粤菜）\n\n### 第三天：自然风光与美食体验\n**上午**：\n- **白云山**（广州名山，登山观景）\n- **云台花园**（春日花海）\n\n**下午**：\n- **长隆野生动物世界**（可选，如时间充裕）\n- **东山口**（文艺街区，特色小店）\n\n**晚上**：\n- **老字号饮茶**（北园酒家、泮溪酒家等）\n- **特色粤菜**（九大簋家宴等）\n\n## 🍽️ 美食推荐\n- **早茶**：北园酒家、泮溪酒家、南园酒家\n- **粤菜**：九大簋家宴、侨美食家\n- **小吃**：荔银肠粉、上下九步行街小吃\n- **甜品**：特色糖水、点心\n\n## 🚇 交通建议\n- 主要使用地铁出行（广州地铁网络发达）\n- 下载"广州地铁"APP方便查询路线\n- 景点间距离较近的可步行或共享单车\n\n## 🎒 出行准备\n- **衣物**：轻薄夏装，带件薄外套防空调\n- **雨具**：雨伞或雨衣（雷阵雨频繁）\n- **防晒**：防晒霜、帽子、太阳镜\n- **药品**：防暑药品、肠胃药\n- **其他**：充电宝、舒适鞋子\n\n## 💡 实用贴士\n1. 避开周末高峰期游览热门景点\n2. 提前预订热门餐厅和景点门票\n3. 注意防暑降温，多补充水分\n4. 雷阵雨多在下午，可安排室内活动\n5. 使用手机支付（微信/支付宝）更方便\n\n祝您在广州玩得愉快！', 'reflection_decision': 'next', 'final_answer': '根据您的需求和查询到的天气信息，我为您整理了一份广州三日游出行计划。以下是结合天气情况、景点推荐和实用建议的详细安排：\n\n---\n\n### 🌤️ **天气情况（10月11日-13日）**\n- **10月11日（周六）**：晴转雷阵雨，35°C/25°C  \n- **10月12日（周日）**：雷阵雨，34°C/24°C  \n- **10月13日（周一）**：雷阵雨，33°C/24°C  \n\n**天气提醒**：  \n- 白天炎热，紫外线较强，需注意防晒。  \n- 下午至夜间多雷阵雨，建议随身携带雨具，并灵活调整户外行程。\n\n---\n\n### 🗺️ **三日游行程规划**\n\n#### **第一天：历史文化与城市精华**\n**上午**：  \n- **越秀公园**（免费）：游览五羊石像、镇海楼，感受广州城市象征。  \n- **中山纪念堂**（10元）：参观标志性建筑，了解孙中山先生的历史事迹。  \n\n**下午**：  \n- **陈家祠**（10元）：欣赏岭南传统建筑的精美雕刻与设计。  \n- **永庆坊**（文艺街区）：体验西关老城风情，品尝老字号小吃。  \n\n**晚上**：  \n- **珠江夜游**（天字码头）：乘船观赏广州塔、猎德大桥等璀璨夜景。  \n- **北京路步行街**：逛千年古道遗址，感受热闹的夜市氛围。\n\n---\n\n#### **第二天：现代都市与欧陆风情**\n**上午**：  \n- **广州塔**（“小蛮腰”）：登塔俯瞰全城景色（建议提前购票）。  \n- **花城广场**：打卡现代建筑群，与广州塔合影。  \n\n**下午**：  \n- **沙面岛**（免费）：漫步欧式建筑群，感受历史与文艺交融。  \n- **圣心大教堂**（免费）：参观石室教堂，注意着装端庄。  \n\n**晚上**：  \n- **上下九步行街**：品尝地道粤菜与小食，如银记肠粉、南信双皮奶。  \n- **大佛寺**（夜景）：欣赏金碧辉煌的仿古建筑灯光。\n\n---\n\n#### **第三天：自然风光与休闲体验**\n**上午**：  \n- **白云山**（门票约5元）：登山观景，呼吸新鲜空气，俯瞰广州全景。  \n- **云台花园**（可选）：欣赏四季花卉与园林景观。  \n\n**下午**：  \n- **广东省博物馆**（免费，需预约）：了解岭南文化与历史。  \n- **东山口**：逛文艺街区、特色小店，感受老洋楼与潮流文化的结合。  \n\n**晚上**：  \n- **老字号饮茶**：推荐北园酒家或泮溪酒家，体验正宗广式早茶点心。  \n- **太古仓码头**（可选）：江边餐厅用餐，欣赏日落与夜景。\n\n---\n\n### 🍽️ **美食推荐**\n- **早茶/点心**：虾饺、烧卖、红米肠（推荐北园酒家、泮溪酒家）。  \n- **粤菜经典**：白切鸡、煲仔饭、清蒸鱼（九大簋家宴、侨美食家）。  \n- **小吃甜品**：双皮奶、肠粉、萝卜牛杂（上下九步行街、南信牛奶甜品）。\n\n---\n\n### 🚇 **交通与贴士**\n1. **出行方式**：  \n   - 优先选择地铁（下载“广州地铁”APP查询路线）。  \n   - 景点间距离近的可步行或骑共享单车。  \n2. **必备物品**：  \n   - 轻便夏装、薄外套（应对室内空调）、雨伞、防晒霜、舒适鞋子。  \n3. **其他建议**：  \n   - 提前预约热门景点（如广东省博物馆）。  \n   - 避开午间高温时段，合理安排室内外活动。  \n   - 使用手机支付（微信/支付宝）更便捷。\n\n---\n\n### 💎 **总结**\n广州是一座融合历史与现代的多元城市，三天的行程可涵盖其文化精髓、自然风光与美食体验。根据天气，建议白天注重防晒，下午备好雨具，灵活调整行程。祝您旅途愉快，尽情享受广州的魅力！\n\n（注：行程可根据个人兴趣和时间灵活调整，如时间充裕可增加长隆野生动物世界等景点。）'}}
t = {'tool': {'question': '我明天想去广东的汕头玩，请你看一下天气情况和准备一下出行计划。', 'need_tools': True, 'retry_count': 2, 'tool_result_history': ['基于查询到的信息，我为您准备了详细的汕头出行计划：\n\n## 汕头出行计划（明天）\n\n### 🌤️ 天气情况\n**明天（10月11日，周六）天气：**\n- **白天：** 晴，最高温度33°C\n- **夜间：** 晴，最低温度27°C  \n- **风向：** 东南风1-3级\n- **天气状况：** 晴朗舒适，适合户外活动\n\n**未来几天天气趋势：**\n- 周日：晴转阵雨，温度27-33°C\n- 周一：晴转多云，温度27-32°C\n\n### 🎯 出行建议\n\n**衣物准备：**\n- 轻薄夏装（短袖、短裤、裙子）\n- 防晒用品（帽子、太阳镜、防晒霜）\n- 舒适的步行鞋\n- 雨伞（周日可能有阵雨）\n\n### 🏞️ 推荐景点\n\n**1. 汕头小公园**\n- 汕头老城核心地标，民国建筑风格\n- 必游：中山纪念亭、南生百货大楼、汕头开埠文化陈列馆\n- 附近美食：小公园蛋挞、爱西干面、侗平肠粉\n\n**2. 东海岸公园**\n- 13公里无敌海岸线，被称为"汕头迈亚密"\n- 四个园区：时间之环、时间溪谷、梦想腾飞、拥抱未来\n- 适合：运动、野餐、放风筝、欣赏日落\n\n**3. 礐石风景区**\n- 汕头最美后花园，自然风光优美\n\n**4. 南澳岛**\n- 海岛风光，青澳湾沙滩\n\n### 🍽️ 美食推荐\n\n**必吃美食：**\n1. **牛肉火锅** - 杏花吴记、八合里、福合埕\n2. **潮汕生腌** - 瑞娇嫲嫲、金二顺\n3. **卤水鹅** - 日日香卤鹅饭店\n4. **粽球** - 小公园老牌粽球\n5. **粿条汤** - 老胡牛肉粿\n\n**特色餐厅：**\n- 杏花吴记牛肉火锅（汕头牛肉火锅第一名）\n- 福合埕牛肉丸（60年老字号）\n- 吴记富苑（消夜打冷天花板）\n- 瑞娇嫲嫲潮汕生腌（明星打卡店）\n\n### 📝 行程建议\n\n**一日游推荐路线：**\n- **上午：** 游览汕头小公园，感受老城风情\n- **中午：** 在小公园附近品尝地道美食\n- **下午：** 前往东海岸公园，欣赏海景和日落\n- **晚上：** 品尝潮汕特色火锅或生腌海鲜\n\n**注意事项：**\n- 天气炎热，注意防晒补水\n- 生腌海鲜初次尝试要适量\n- 提前规划交通，汕头公交系统发达\n- 热门餐厅可能需要排队，建议错峰用餐\n\n祝您在汕头玩得愉快！', '基于查询到的信息，我为您准备了详细的汕头出行计划：\n\n## 🌤️ 汕头天气情况（明天）\n\n**日期：2025年10月11日（星期六）**\n- **天气状况：晴**\n- **气温：27°C - 33°C**\n- **风向风力：东南风 1-3级**\n- **紫外线：较强，建议做好防晒**\n\n**未来几天天气趋势：**\n- 10月12日：晴转阵雨，27-33°C\n- 10月13日：晴转多云，27-32°C\n\n## 🗺️ 汕头出行计划\n\n### 🏛️ 必游景点推荐\n\n**1. 汕头小公园历史文化街区**\n- 民国建筑群，中山纪念亭为核心\n- 骑楼老街、老字号商铺\n- 附近美食：小公园蛋挞、爱西干面、侗平肠粉\n\n**2. 南澳岛**\n- 美丽海岛风光，青澳湾沙滩\n- 南澳大桥壮观景色\n- 适合海边休闲、拍照打卡\n\n**3. 礐石风景区**\n- 汕头最美后花园\n- 自然风光优美，适合徒步\n\n**4. 东海岸公园**\n- 13公里无敌海岸线\n- 四个园区：时间之环、时间溪谷、梦想腾飞、拥抱未来\n- 适合放风筝、野餐、看日落\n\n**5. 菩提禅寺**\n- 被誉为"广东版布达拉宫"\n- 万佛楼、观音溶洞\n- 免费参观，夜景灯光很美\n\n### 🍽️ 美食攻略\n\n**必吃美食：**\n- **牛肉火锅**：杏花吴记、八合里、福合埕\n- **卤水鹅**：日日香卤鹅饭店\n- **生腌海鲜**：瑞娇嫲嫲\n- **潮汕打冷**：吴记富苑\n- **特色小吃**：老牌粽球、牛肉粿条、肠粉\n\n**推荐餐厅：**\n1. 杏花吴记牛肉火锅（汕头牛肉火锅第一名）\n2. 八合里牛肉火锅（总店）\n3. 福合埕牛肉丸（60年老字号）\n4. 吴记富苑（消夜打冷天花板）\n5. 小公园老牌粽球（传统风味）\n\n### 📝 出行建议\n\n**穿着建议：**\n- 轻便夏装，带薄外套\n- 舒适运动鞋（景点较多需要步行）\n- 防晒用品：帽子、太阳镜、防晒霜\n\n**交通建议：**\n- 市内公交便利，覆盖主要景点\n- 可考虑租车或使用网约车\n- 南澳岛需过桥，建议安排一整天\n\n**行程安排建议：**\n- **上午**：游览汕头小公园历史文化街区\n- **中午**：品尝正宗潮汕牛肉火锅\n- **下午**：前往东海岸公园或礐石风景区\n- **晚上**：体验潮汕打冷和生腌海鲜\n\n**温馨提示：**\n- 天气炎热，注意补水防晒\n- 美食较多，建议分餐品尝\n- 提前预订热门餐厅，避免排队\n- 南澳岛需提前规划交通\n\n祝您在汕头玩得愉快！'], 'tool_result': '基于查询到的信息，我为您准备了详细的汕头出行计划：\n\n## 🌤️ 汕头天气情况（明天）\n\n**日期：2025年10月11日（星期六）**\n- **天气状况：晴**\n- **气温：27°C - 33°C**\n- **风向风力：东南风 1-3级**\n- **紫外线：较强，建议做好防晒**\n\n**未来几天天气趋势：**\n- 10月12日：晴转阵雨，27-33°C\n- 10月13日：晴转多云，27-32°C\n\n## 🗺️ 汕头出行计划\n\n### 🏛️ 必游景点推荐\n\n**1. 汕头小公园历史文化街区**\n- 民国建筑群，中山纪念亭为核心\n- 骑楼老街、老字号商铺\n- 附近美食：小公园蛋挞、爱西干面、侗平肠粉\n\n**2. 南澳岛**\n- 美丽海岛风光，青澳湾沙滩\n- 南澳大桥壮观景色\n- 适合海边休闲、拍照打卡\n\n**3. 礐石风景区**\n- 汕头最美后花园\n- 自然风光优美，适合徒步\n\n**4. 东海岸公园**\n- 13公里无敌海岸线\n- 四个园区：时间之环、时间溪谷、梦想腾飞、拥抱未来\n- 适合放风筝、野餐、看日落\n\n**5. 菩提禅寺**\n- 被誉为"广东版布达拉宫"\n- 万佛楼、观音溶洞\n- 免费参观，夜景灯光很美\n\n### 🍽️ 美食攻略\n\n**必吃美食：**\n- **牛肉火锅**：杏花吴记、八合里、福合埕\n- **卤水鹅**：日日香卤鹅饭店\n- **生腌海鲜**：瑞娇嫲嫲\n- **潮汕打冷**：吴记富苑\n- **特色小吃**：老牌粽球、牛肉粿条、肠粉\n\n**推荐餐厅：**\n1. 杏花吴记牛肉火锅（汕头牛肉火锅第一名）\n2. 八合里牛肉火锅（总店）\n3. 福合埕牛肉丸（60年老字号）\n4. 吴记富苑（消夜打冷天花板）\n5. 小公园老牌粽球（传统风味）\n\n### 📝 出行建议\n\n**穿着建议：**\n- 轻便夏装，带薄外套\n- 舒适运动鞋（景点较多需要步行）\n- 防晒用品：帽子、太阳镜、防晒霜\n\n**交通建议：**\n- 市内公交便利，覆盖主要景点\n- 可考虑租车或使用网约车\n- 南澳岛需过桥，建议安排一整天\n\n**行程安排建议：**\n- **上午**：游览汕头小公园历史文化街区\n- **中午**：品尝正宗潮汕牛肉火锅\n- **下午**：前往东海岸公园或礐石风景区\n- **晚上**：体验潮汕打冷和生腌海鲜\n\n**温馨提示：**\n- 天气炎热，注意补水防晒\n- 美食较多，建议分餐品尝\n- 提前预订热门餐厅，避免排队\n- 南澳岛需提前规划交通\n\n祝您在汕头玩得愉快！', 'reflection_decision': '{"reflection_decision": "retry", "reflection_suggestion": "需要重新查询天气信息，因为当前回答中的日期（10月11日）与用户提问的\'明天\'时间不匹配，这可能是使用了过时的缓存数据。应该重新调用天气查询工具获取准确的明日天气预报，包括温度、天气状况、降水概率等关键信息，确保出行建议的准确性。"}'}}
(s["summarize"].keys(), t["tool"].keys())

(dict_keys(['question', 'need_tools', 'retry_count', 'tool_result_history', 'tool_result', 'reflection_decision', 'final_answer']),
 dict_keys(['question', 'need_tools', 'retry_count', 'tool_result_history', 'tool_result', 'reflection_decision']))

In [42]:
print("="*80,"\n")
print("="*80,"\n")